In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner 
import string
import random
import datetime
import matplotlib.pyplot as plt
import shutil                      # for deleting folders with files in them 
import os

In [3]:
from tensorflow.python.client import device_lib
def get_available_devices():
    local_device_protos = device_lib.list_local_devices()
    return [x.name for x in local_device_protos]
print(get_available_devices()) 

['/device:CPU:0']


In [4]:
import sys
print (sys.version)
# 3.6.4 |Anaconda custom (64-bit)| (default, Jan 16 2018, 10:22:32) [MSC v.1900 64 bit (AMD64)]
import tensorflow as tf
print(tf.__version__)

3.7.8 (tags/v3.7.8:4b47a5b6ba, Jun 28 2020, 08:53:46) [MSC v.1916 64 bit (AMD64)]
2.5.0


In [5]:
# tensorflow 2.5 needs cuDNN 8.1	CUDA 11.2

negative sampling: https://ieeexplore.ieee.org/document/9010476

keras code reference: https://keras.io/examples/structured_data/collaborative_filtering_movielens/

https://keras.io/guides/training_with_built_in_methods/

Hyperparameter tuning: I ended up going with https://www.tensorflow.org/tutorials/keras/keras_tuner rather than https://www.tensorflow.org/tensorboard/hyperparameter_tuning_with_hparams

in my use case with real data, the model kept recommending the same items to everyone, so I made the model select negative examples with probability the inverse of each item's popularity (so that more rare items are upweighted). 

In [3]:
# function which outputs a random alpha-numeric string of specified length:
def random_string(nchar):
    available_characters = string.ascii_lowercase+''.join([str(i) for i in range(10)])
    return ''.join( random.choice(available_characters) for i in range(nchar) )
    
# example use:
random_string(20)

'nrf9cbtpvb411k7xilyk'

In [5]:
# each item belongs to exactly one item group (items are distributed uniformly across the item groups)
# each user will only consume items from exactly one item group
n_unique_users = 1000
n_unique_items = 50
n_item_groups = 6
n_items_per_user = 5

user_IDs = [ 'user_' + str(i) + '_' + random_string(10) for i in range(n_unique_users) ]
item_IDs = [ 'item_' + str(i) + '_' + random_string(10) for i in range(n_unique_items) ]

# assign each item to an item group:
item_group_per_item_ref = {}
for i in range(n_item_groups):
    item_group_per_item_ref[f'item_group_{i+1}'] = []

for item_i in item_IDs:
    item_group = random.choice( list(item_group_per_item_ref.keys()) )
    item_group_per_item_ref[item_group].append( item_i )

# store the simulated item population in a dataframe: 
store_intermediate_itemgroup_dfs = []
for item_group_i in item_group_per_item_ref.keys():
    item_IDs_in_group_i = item_group_per_item_ref[item_group_i]
    n_items_in_group_i = len(item_IDs_in_group_i)
    relative_unscaled_item_probs_group_i = np.random.exponential( scale=1, size=n_items_in_group_i )
    store_intermediate_itemgroup_dfs.append( 
        pd.DataFrame(
            {
                'item_group':[item_group_i] * n_items_in_group_i,
                'item_ID':item_IDs_in_group_i,
                'selection_prob': relative_unscaled_item_probs_group_i / sum(relative_unscaled_item_probs_group_i)
            }
        )
    )
item_ref_df = pd.concat( store_intermediate_itemgroup_dfs, axis=0 )    
    
gendata_list = []
user_counter = 0
for user_i in range(n_unique_users):
    if user_counter%1000==0:
        print(user_counter)
    # give each user [n_items_per_user] items that they did buy (positive examples), and [n_items_per_user] items that they didn't buy (negative examples)
    # choose an item group for this user:
    item_group_this_user = random.choice( list( item_group_per_item_ref.keys() ) )  
    items_bought_by_this_user = np.random.choice( 
                                     # items bought must be from their item group                          
                                    a = item_ref_df.query( f'item_group=="{item_group_this_user}"' )['item_ID'].values ,
                                    size = n_items_per_user,
                                    replace = False,
                                    p = item_ref_df.query( f'item_group=="{item_group_this_user}"' )['selection_prob'].values 
                                ).tolist()
    gendata_list.append(
        pd.DataFrame(
            {
                'user_ID':[user_IDs[user_i]]*n_items_per_user,
                'item_ID':items_bought_by_this_user
            }
        )
    )
    #items_not_bought_by_this_user_df = (
    #    item_ref_df
    #    .copy()
    #    .merge( pd.DataFrame({'item_ID':items_bought_by_this_user,'remove':[1]*len(items_bought_by_this_user)}),
    #            how = 'left',
    #            on = 'item_ID'
    #          )
    #    .query('remove.isnull()', engine='python')
    #)
    #items_not_bought_by_this_user_df['selection_prob'] = items_not_bought_by_this_user_df['selection_prob'] / items_not_bought_by_this_user_df['selection_prob'].sum() 
    #random_negative_examples = np.random.choice(                      
    #                                a = items_not_bought_by_this_user_df['item_ID'].values ,
    #                                size = n_items_per_user,
    #                                replace = False,
    #                                p = items_not_bought_by_this_user_df['selection_prob'].values 
    #                            ).tolist()
    #gendata_list.append(
    #    pd.DataFrame(
    #        {
    #            'user_ID':[user_IDs[user_i]]*n_items_per_user*2,
    #            'item_ID':items_bought_by_this_user + random_negative_examples,
    #            'y':[1]*n_items_per_user + [0]*n_items_per_user 
    #        }
    #    )
    #)
    user_counter += 1

full_dataset = pd.concat(gendata_list, axis=0).reset_index()
print( full_dataset.shape )
full_dataset.head(20)

0
(5000, 3)


,index,user_ID,item_ID
0,0,user_0_hmullvm0wd,item_4_drgnl0tu0t
1,1,user_0_hmullvm0wd,item_25_yp69xqno6k
2,2,user_0_hmullvm0wd,item_28_aiandulwnh
3,3,user_0_hmullvm0wd,item_34_rx6vsir4z1
4,4,user_0_hmullvm0wd,item_12_5pmhj9wqq2
5,0,user_1_ephrclbmkm,item_4_drgnl0tu0t
6,1,user_1_ephrclbmkm,item_25_yp69xqno6k
7,2,user_1_ephrclbmkm,item_34_rx6vsir4z1
8,3,user_1_ephrclbmkm,item_3_0cs61gfkxo
9,4,user_1_ephrclbmkm,item_12_5pmhj9wqq2


In [6]:
# look at the distribution of item sales in the data:
item_bought_counts = full_dataset[['user_ID','item_ID']].copy().groupby('item_ID').agg(n_bought=('user_ID','count')).reset_index()
# make the distribution of negative examples match the item distribution in the positive examples:
# (this hopefully stops the recommender from just recommending the same items to everyone)
# previously, tried weighting by inverse popularity, but this didn't solve the problem
item_bought_counts['item_selection_prob'] = item_bought_counts['n_bought']
item_bought_counts['item_selection_prob'] = item_bought_counts['item_selection_prob'] / item_bought_counts['item_selection_prob'].sum()

item_bought_counts

,item_ID,n_bought,item_selection_prob
0,item_0_8rb7spaxzw,7,0.0014
1,item_10_959tj0bgrg,124,0.0248
2,item_11_kilcjhg2ky,161,0.0322
3,item_12_5pmhj9wqq2,150,0.0300
4,item_13_3vljltvtkd,167,0.0334
5,item_14_4j2a1e2ff0,80,0.0160
6,item_15_zc6sp5jgyv,161,0.0322
7,item_16_fij8dj9l4c,113,0.0226
8,item_17_f6bh5kogg1,69,0.0138
9,item_18_wxal8b803h,30,0.0060


In [7]:
# make a list of unique user_IDs and item_IDs:
user_IDs = full_dataset['user_ID'].drop_duplicates().values
item_IDs = full_dataset['item_ID'].drop_duplicates().values
user_IDs[0:5]
item_IDs[0:5]

array(['item_4_drgnl0tu0t', 'item_25_yp69xqno6k', 'item_28_aiandulwnh',
       'item_34_rx6vsir4z1', 'item_12_5pmhj9wqq2'], dtype=object)

In [8]:
# give each customer as many negative examples (items not bought) as they have positive examples:
full_dataset['y'] = 1    # all actual items bought are positive examples
negative_examples_temp_storage_list = []
counter = 0
for user_i in user_IDs:
    items_bought = full_dataset.query(f'user_ID=="{user_i}"')['item_ID'].values.tolist()
    items_not_bought = (          #[i for i in item_IDs if i not in items_bought]
        item_bought_counts
        .merge( 
                pd.DataFrame( {'item_ID':items_bought,'bought_flag':[1]*len(items_bought)} ),
                how = 'left',
                on = 'item_ID'
        )
        .query('bought_flag.isnull()', engine='python')   
    )
    negative_examples = np.random.choice( 
                            items_not_bought['item_ID'].values, 
                            p = items_not_bought['item_selection_prob'].values / items_not_bought['item_selection_prob'].sum(),
                            size = min( [len(items_bought), len(items_not_bought)] ),
                            replace = False
                        )
    negative_examples_temp_storage_list.append(
        pd.DataFrame( {
            'user_ID':[user_i]*len(negative_examples),
            'item_ID':negative_examples,
            'y':[0]*len(negative_examples)
        }
        )
    )
    # print progress to screen:
    print( f'{round( 100 * counter / len(user_IDs), 2 )}%' )
    counter += 1

# add the negative examples to the original dataset:
full_dataset = full_dataset.append( pd.concat(negative_examples_temp_storage_list, axis=0) )
full_dataset = full_dataset.sort_values(['user_ID','item_ID']).reset_index()[['user_ID','item_ID','y']]

full_dataset.sort_values('user_ID')

0.0%
0.1%
0.2%
0.3%
0.4%
0.5%
0.6%
0.7%
0.8%
0.9%
1.0%
1.1%
1.2%
1.3%
1.4%
1.5%
1.6%
1.7%
1.8%
1.9%
2.0%
2.1%
2.2%
2.3%
2.4%
2.5%
2.6%
2.7%
2.8%
2.9%
3.0%
3.1%
3.2%
3.3%
3.4%
3.5%
3.6%
3.7%
3.8%
3.9%
4.0%
4.1%
4.2%
4.3%
4.4%
4.5%
4.6%
4.7%
4.8%
4.9%
5.0%
5.1%
5.2%
5.3%
5.4%
5.5%
5.6%
5.7%
5.8%
5.9%
6.0%
6.1%
6.2%
6.3%
6.4%
6.5%
6.6%
6.7%
6.8%
6.9%
7.0%
7.1%
7.2%
7.3%
7.4%
7.5%
7.6%
7.7%
7.8%
7.9%
8.0%
8.1%
8.2%
8.3%
8.4%
8.5%
8.6%
8.7%
8.8%
8.9%
9.0%
9.1%
9.2%
9.3%
9.4%
9.5%
9.6%
9.7%
9.8%
9.9%
10.0%
10.1%
10.2%
10.3%
10.4%
10.5%
10.6%
10.7%
10.8%
10.9%
11.0%
11.1%
11.2%
11.3%
11.4%
11.5%
11.6%
11.7%
11.8%
11.9%
12.0%
12.1%
12.2%
12.3%
12.4%
12.5%
12.6%
12.7%
12.8%
12.9%
13.0%
13.1%
13.2%
13.3%
13.4%
13.5%
13.6%
13.7%
13.8%
13.9%
14.0%
14.1%
14.2%
14.3%
14.4%
14.5%
14.6%
14.7%
14.8%
14.9%
15.0%
15.1%
15.2%
15.3%
15.4%
15.5%
15.6%
15.7%
15.8%
15.9%
16.0%
16.1%
16.2%
16.3%
16.4%
16.5%
16.6%
16.7%
16.8%
16.9%
17.0%
17.1%
17.2%
17.3%
17.4%
17.5%
17.6%
17.7%
17.8%
17.9%
18.0%
18.1%
18.2%
18

,user_ID,item_ID,y
0,user_0_hmullvm0wd,item_11_kilcjhg2ky,0
1,user_0_hmullvm0wd,item_12_5pmhj9wqq2,1
2,user_0_hmullvm0wd,item_13_3vljltvtkd,0
3,user_0_hmullvm0wd,item_25_yp69xqno6k,1
4,user_0_hmullvm0wd,item_28_aiandulwnh,1
...,...,...,...
9994,user_9_4akavncwov,item_33_jyrjy7k3l6,1
9995,user_9_4akavncwov,item_36_b41s16otwt,1
9996,user_9_4akavncwov,item_43_y2aj7gndck,0
9997,user_9_4akavncwov,item_44_52b6raiyci,1


In [9]:
positive_examples_population_calc = (
    full_dataset
    .query('y==1')
    [['item_ID']]
    .groupby('item_ID')
    .agg( nRows = ('item_ID','count') )
    .reset_index()
)
negative_examples_population_calc = (
    full_dataset
    .query('y==0')
    [['item_ID']]
    .groupby('item_ID')
    .agg( nRows = ('item_ID','count') )
    .reset_index()
)

#positive_examples_population_calc.merge(
#        
#        how = 'full_outer'
#    )

In [10]:
#full_dataset = pd.read_csv( "C://Users//jbolton//Documents//blogs//collab_filter_logs//full_dataset_temp.csv" )
#user_IDs = full_dataset['user_ID'].drop_duplicates().values
#item_IDs = full_dataset['item_ID'].drop_duplicates().values

In [11]:
# assign integer IDs to the users and items:
# (a one-to-one mapping of the existing user_IDs and item_IDs onto integers)
user2user_encoded = {x: i for i, x in enumerate(user_IDs)}     # mapping from user_ID to integer user ID
userencoded2user = {i: x for i, x in enumerate(user_IDs)}      # mapping from integer user ID to user_ID

item2item_encoded = {x: i for i, x in enumerate(item_IDs)}    # mapping from item_ID to integer item ID
item_encoded2item = {i: x for i, x in enumerate(item_IDs)}    # mapping from integer item ID to item_ID

# add the integer versions of the user and item IDs to the dataset:
full_dataset["int_user_ID"] = full_dataset["user_ID"].map(user2user_encoded)
full_dataset["int_item_ID"] = full_dataset["item_ID"].map(item2item_encoded)

In [16]:
validation_indices = []         # to store which rows of the data are going to go to the validation set 
test_indices = []               # to store which rows of the data are going to go to the test set
counter = 0
for userID_i in user_IDs:
    positive_example_row_indices = full_dataset.index[ (full_dataset['user_ID']==userID_i) & (full_dataset['y']==1) ].tolist()
    negative_example_row_indices = full_dataset.index[ (full_dataset['user_ID']==userID_i) & (full_dataset['y']==0) ].tolist()
    indices_of_2_positive_examples = random.sample(positive_example_row_indices, k=2)
    indices_of_2_negative_examples = random.sample(negative_example_row_indices, k=2)
    validation_indices.append( indices_of_2_positive_examples[0] )
    validation_indices.append( indices_of_2_negative_examples[0] )
    test_indices.append( indices_of_2_positive_examples[1] )
    test_indices.append( indices_of_2_negative_examples[1] )
    # print progress to screen:
    #if round( 100 * counter / len(user_IDs), 2 ) % 5 == 0:
    #    print( round( 100 * counter / len(user_IDs), 2 ) )
    #counter += 1

# train_indices = [ i for i in list(full_dataset.index) if i not in set(validation_indices+test_indices) ]   # this takes too long! 
all_indices_df = pd.DataFrame(
    {
        'myindex':list(full_dataset.index)
    }
)
index_to_remove_df = pd.DataFrame(
    {
        'myindex':list(validation_indices+test_indices),
        'remove':[1]*len( list(validation_indices+test_indices) )
    }
)
train_indices = (
    all_indices_df
    .merge( index_to_remove_df, how='left', on='myindex' )
    .query( 'remove.isnull()', engine='python')
    ['myindex']
    .values
)

training_data = full_dataset.iloc[ train_indices ].copy()
validation_data = full_dataset.iloc[ validation_indices ].copy()
test_data = full_dataset.iloc[ test_indices ].copy()

print( 
f"""
nrows total data: {len(full_dataset)}
nrows training data: {len(training_data)}
nrows validation data: {len(validation_data)}
nrows test data: {len(test_data)}
"""
)


nrows total data: 10000
nrows training data: 6000
nrows validation data: 2000
nrows test data: 2000



In [13]:
# prepare model inputs:
#x = full_sim_dataset[['int_user_ID', 'int_item_ID']].values
#y = full_sim_dataset['y'].values
#x_train, x_val, x_test, y_train, y_val, y_test = (
#    x[train_indices],      # x_train
#    x[validation_indices], # x_val
#    x[test_indices],       # x_test
#    y[train_indices],      # y_train
#    y[validation_indices], # y_val
#    y[test_indices]       # y_test
#)

In [18]:
user_IDs

array(['user_0_hmullvm0wd', 'user_1_ephrclbmkm', 'user_2_o4kbob5189',
       'user_3_4k6vr5n2rj', 'user_4_85xhxof6k1', 'user_5_5ylhcsrznw',
       'user_6_hfpakonuwn', 'user_7_fm9c5e4xfz', 'user_8_94r2g3pemn',
       'user_9_4akavncwov', 'user_10_rpm5k08ubx', 'user_11_i77gta7hyo',
       'user_12_wd76at9fde', 'user_13_1l405bc3a4', 'user_14_yixfnozya0',
       'user_15_7v5hec1095', 'user_16_3bcoz1ggpl', 'user_17_o520s5xrzp',
       'user_18_xykx4kxsya', 'user_19_8twyvpe28u', 'user_20_vssves7udg',
       'user_21_no1g4ujeg9', 'user_22_73lpisqmfa', 'user_23_n86l663ett',
       'user_24_vnofpm81u8', 'user_25_whuv2r2qma', 'user_26_chapfm4h26',
       'user_27_0dr1mgwfu3', 'user_28_porp16wyjm', 'user_29_s5ehn7xlhu',
       'user_30_mikdntk292', 'user_31_plxpf1te89', 'user_32_q3v0jkqbpb',
       'user_33_drr0dlzij2', 'user_34_fiicqpsfj9', 'user_35_rqt17ttvz5',
       'user_36_u1kjtxzzil', 'user_37_incr4g2bcu', 'user_38_xq6bntpyov',
       'user_39_3j8lbh9uqa', 'user_40_nrxw8ibsel', 'user_41_6

In [17]:
# quick error check: check what has happened with a random user's data:
random_user_ID = random.choice( user_IDs )
print( training_data[['user_ID','int_item_ID','y']].query( f'user_ID=={random_user_ID}') )
print( validation_data[['user_ID','int_item_ID','y']].query( f'user_ID=={random_user_ID}') )
print( test_data[['user_ID','int_item_ID','y']].query( f'user_ID=={random_user_ID}') )
print( full_dataset[['user_ID','int_item_ID','y']].query( f'user_ID=={random_user_ID}') )

UndefinedVariableError: name 'user_230_56m8q6c1ew' is not defined

In [ ]:
# prepare model inputs:
x = full_dataset[['int_user_ID', 'int_item_ID']].values
y = full_dataset['y'].values
x_train, x_val, x_test, y_train, y_val, y_test = (
    x[train_indices],      # x_train
    x[validation_indices], # x_val
    x[test_indices],       # x_test
    y[train_indices],      # y_train
    y[validation_indices], # y_val
    y[test_indices]       # y_test
)

In [ ]:
class RecommenderNet(keras.Model):
    def __init__(self, num_users, num_items, embedding_size, embedding_L2_regularizer, **kwargs):
        super(RecommenderNet, self).__init__(**kwargs)
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_size = embedding_size
        self.user_embedding = layers.Embedding(
            num_users,
            embedding_size,
            embeddings_initializer = "he_normal",
            embeddings_regularizer = keras.regularizers.l2( embedding_L2_regularizer ) # keras.regularizers.l2(1e-6),
        )
        self.user_bias = layers.Embedding(num_users, 1)
        self.item_embedding = layers.Embedding(
            num_items,
            embedding_size,
            embeddings_initializer = "he_normal",
            embeddings_regularizer = keras.regularizers.l2( embedding_L2_regularizer ) # keras.regularizers.l2(1e-6),
        )
        self.item_bias = layers.Embedding(num_items, 1)

    def call(self, inputs):
        user_vector = self.user_embedding(inputs[:, 0])
        user_bias = self.user_bias(inputs[:, 0])
        item_vector = self.item_embedding(inputs[:, 1])
        item_bias = self.item_bias(inputs[:, 1])
        dot_user_item = tf.tensordot(user_vector, item_vector, 2)
        # Add all the components (including bias)
        x = dot_user_item + user_bias + item_bias
        # The sigmoid activation forces the rating to between 0 and 1
        return tf.nn.sigmoid(x)

#model = RecommenderNet( 
#    num_users = len(user_IDs), 
#    num_items = len(item_IDs), 
#    embedding_size = EMBEDDING_SIZE 
#)
#model.compile(
#    loss = tf.keras.losses.BinaryCrossentropy(),
#    optimizer = keras.optimizers.Adam(lr=0.001)
#)

def model_builder(hp):
    model = RecommenderNet( 
                num_users = len(user_IDs), 
                num_items = len(item_IDs), 
                embedding_size = hp.Choice( 'embedding_size', values=[5,10,50,100] ),
                embedding_L2_regularizer = hp.Choice('embedding_L2_regularizer', values=[0.0, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1])
    )
    model.compile(
                    loss = tf.keras.losses.BinaryCrossentropy(),
                    optimizer = keras.optimizers.Adam( hp.Choice( 'learning_rate', values=[0.0001, 0.001, 0.01, 0.1] ) )
                )
    return model

In [ ]:
# make my own Keras tuner so that 'batch_size' can also be tuned: 
class MyTuner( keras_tuner.tuners.Hyperband ):
  def run_trial(self, trial, *args, **kwargs):
    # You can add additional HyperParameters for preprocessing and custom training loops
    # via overriding `run_trial`
    kwargs['batch_size'] = trial.hyperparameters.Int('batch_size', 500, 10000, step=500)
    super(MyTuner, self).run_trial(trial, *args, **kwargs)

In [ ]:
# delete current model logs (so that tuning does not continue from previous tuner):
tuner_logs_folder_path = 'C://users//jbolton//Documents//blogs//collab_filter_logs//hyperband_logs//' 
if os.path.isdir( tuner_logs_folder_path ):   # if this folder already exists 
    shutil.rmtree( tuner_logs_folder_path )   # ..then delete it

# define hyperparameter tuner:
tuner = MyTuner(
            model_builder,
            objective = 'val_loss',
            max_epochs = 100,
            factor = 3,
            directory = 'C://users//jbolton//Documents//blogs//collab_filter_logs//hyperband_logs//',
            project_name = 'collab_filter_recommender'
)
#tuner = keras_tuner.Hyperband(
#            model_builder,
#            objective = 'val_loss',
#            max_epochs = 50,
#            factor = 3,
#            directory = 'C://users//jbolton//Documents//blogs//collab_filter_logs//hyperband_logs//',
#            project_name = 'collab_filter_recommender'
#)

In [ ]:
tensorboardLogs_dir = "C://users//jbolton//Documents//blogs//collab_filter_logs//" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard( 
                            log_dir = tensorboardLogs_dir, 
                            histogram_freq = 1
)

early_stopping_monitor = tf.keras.callbacks.EarlyStopping(
    # see https://keras.io/api/callbacks/early_stopping/
    monitor = 'val_loss',  # quantity to monitor
    min_delta = 0,         # an absolute change of less than min_delta, will count as no improvement
    patience = 20,         # Number of epochs with no improvement after which training will be stopped
    verbose = 1,
    mode = 'auto',
    baseline = None,
    restore_best_weights = True # if True, restores the model weights from the epoch with the best value of the monitored quantity
)

tuner.search(
                x = x_train,
                y = y_train,
                epochs = 100,
                verbose = 1,
                validation_data = (x_val, y_val),
                callbacks = [early_stopping_monitor]
)


In [ ]:
# fetch optimal hyperparameters found:
best_hps = tuner.get_best_hyperparameters( num_trials=1 )[0]
print( f"best embedding_size: {best_hps.get('embedding_size')}" )
print( f"best embedding_L2_regularizer: {best_hps.get('embedding_L2_regularizer')}" )
print( f"best learning_rate: {best_hps.get('learning_rate')}" )
print( f"best batch_size: {best_hps.get('batch_size')}" )

# fetch the model with the best hyperparameters found:
model = tuner.hypermodel.build(best_hps)

In [ ]:
train_history = model.fit(
    x = x_train,
    y = y_train,
    batch_size = best_hps.get('batch_size'),
    epochs = 100,
    verbose = 1,
    validation_data = (x_val, y_val),
    callbacks = [early_stopping_monitor],
)

In [ ]:
plt.plot(train_history.history["loss"])
plt.plot(train_history.history["val_loss"])
plt.title("model loss (binary cross entropy)")
plt.ylabel("loss")
plt.xlabel("epoch")
plt.legend(["training set", "validation set"], loc="upper left")
plt.show()

In [ ]:
test_data['predicted_y'] = [ i[0] for i in model.predict(x_test) ]
test_data

In [ ]:
# calculate the proportion of customers for which the correct product had higher predicted_y: 
evaluate_test_set_performance = (
    test_data[['user_ID','y','predicted_y']]
    .pivot_table( 
                    index = 'user_ID',
                    columns = 'y',
                    values = 'predicted_y'
                )
    .reset_index()
)
evaluate_test_set_performance['correct_decision'] = 0
evaluate_test_set_performance.loc[ evaluate_test_set_performance[1]>evaluate_test_set_performance[0] ,'correct_decision'] = 1

print( f"% of customers with correct decision: {100*evaluate_test_set_performance['correct_decision'].sum() / evaluate_test_set_performance.shape[0]}%" )

In [ ]:
# generate top n recommendations for a random customer:
n_items = 10
random_user_ID = random.choice( user_IDs )
items_bought_in_training_data = training_data.loc[ training_data['user_ID']==random_user_ID, 'item_ID'].values.tolist()
# generate a string nicely formatted to show the items that this random customer bought in the training data:
purchase_string = ''
for i in items_bought_in_training_data:
    purchase_string = purchase_string + '\n\t o  ' + i  
all_products_this_customer_for_model_pred = pd.DataFrame(
    {
        'int_user_ID':[ user2user_encoded[random_user_ID] ] * len(item2item_encoded),
        'int_item_ID':item2item_encoded.values()        # all (integer) item_IDs
    }
).values
model_predictions_all_items_this_random_customer = model.predict( all_products_this_customer_for_model_pred )
model_predictions_every_item_this_random_customer_df = pd.DataFrame(
    {
        'user_ID':[random_user_ID] * len(item2item_encoded),
        'item_ID':item2item_encoded.keys(),
        'predicted_y':[ i[0] for i in model_predictions_all_items_this_random_customer ]
    }
).sort_values( 'predicted_y', ascending=False )
recommended_products_string = ''
for i in model_predictions_every_item_this_random_customer_df.head(n_items)['item_ID'].values:
    recommended_products_string = recommended_products_string + '\n\t *  ' + i  
print(
    f"""
    -- customer_ID {random_user_ID} --
    customer bought in training data:
    {purchase_string}
    \ntop {n_items} recommended products to this customer:
    {recommended_products_string}
    """
)